In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Read Data

df = pd.read_csv("../data/jobs_dataset_with_features.csv")

In [3]:
# Some Info About Data

print("Show First 5 Samples of data:")
print(df.head())

print(f"Shape of data is: {df.shape}")

print("Different Categories in data:")
print(df["Features"].value_counts())

Show First 5 Samples of data:
                        Role  \
0       Social Media Manager   
1     Frontend Web Developer   
2    Quality Control Manager   
3  Wireless Network Engineer   
4         Conference Manager   

                                            Features  
0  5 to 15 Years Digital Marketing Specialist M.T...  
1  2 to 12 Years Web Developer BCA HTML, CSS, Jav...  
2  0 to 12 Years Operations Manager PhD Quality c...  
3  4 to 11 Years Network Engineer PhD Wireless ne...  
4  1 to 12 Years Event Manager MBA Event planning...  
Shape of data is: (1615940, 2)
Different Categories in data:
Features
5 to 13 Years Executive Assistant MBA Office administration and management Budgeting and expense tracking Team coordination and supervision Facility management Vendor and supplier relations Oversee office operations, including facilities management and vendor relations. Manage office budgets and expenses. Coordinate office events and logistics. 105975                        

In [4]:
# Check Null values

print("Null values:")
print(df.isnull().sum())

Null values:
Role        0
Features    0
dtype: int64


In [5]:
# Drop Small Rows Roles

min_count = 6500
role_counts = df["Role"].value_counts()
dropped_classes = role_counts[role_counts < min_count].index
filtered_df = df[~df["Role"].isin(dropped_classes)].reset_index(drop = True)

print(f"Number of Roles is: {len(filtered_df["Role"].value_counts())}")
print(f"Number of Samples is: {filtered_df["Role"].value_counts().sum()}")

Number of Roles is: 61
Number of Samples is: 520692


In [6]:
# Select only some samples

df_sample = filtered_df.sample(n = 50000)

print(f"Number of Roles is: {len(df_sample["Role"].value_counts())}")
print(f"Number of Samples is: {df_sample["Role"].value_counts().sum()}")

Number of Roles is: 61
Number of Samples is: 50000


In [7]:
# Clean Text

import re

def cleanResume(txt):
  # To remove Links
  cleanText = re.sub("http\S+\s", " ", txt)
  # To remove RT, cc
  cleanText = re.sub("RT|cc", " ", cleanText)
  # To remove Hashtags
  cleanText = re.sub("#\S+\s", " ", cleanText)
  # To remove Mentions
  cleanText = re.sub("#@\S+", " ", cleanText)
  # To remove Punctutations
  cleanText = re.sub("[%s]" % re.escape(""""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), " ", cleanText)
  # To remove Non-ASCII
  cleanText = re.sub(r'[^\x00-\x7f]', " ", cleanText)
  # To remove Spaces
  cleanText = re.sub(r'\s+', " ", cleanText)

  return cleanText

<>:7: SyntaxWarning: invalid escape sequence '\S'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:13: SyntaxWarning: invalid escape sequence '\S'
<>:15: SyntaxWarning: invalid escape sequence '\]'
<>:7: SyntaxWarning: invalid escape sequence '\S'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:13: SyntaxWarning: invalid escape sequence '\S'
<>:15: SyntaxWarning: invalid escape sequence '\]'
C:\Users\Moham\AppData\Local\Temp\ipykernel_4124\1606045969.py:7: SyntaxWarning: invalid escape sequence '\S'
  cleanText = re.sub("http\S+\s", " ", txt)
C:\Users\Moham\AppData\Local\Temp\ipykernel_4124\1606045969.py:11: SyntaxWarning: invalid escape sequence '\S'
  cleanText = re.sub("#\S+\s", " ", cleanText)
C:\Users\Moham\AppData\Local\Temp\ipykernel_4124\1606045969.py:13: SyntaxWarning: invalid escape sequence '\S'
  cleanText = re.sub("#@\S+", " ", cleanText)
C:\Users\Moham\AppData\Local\Temp\ipykernel_4124\1606045969.py:15: SyntaxWarning: invalid escape sequence '\]'
  cleanText = 

In [8]:
# Clean Resume

df_sample["Features"] = df_sample["Features"].apply(lambda x: cleanResume(x))

In [9]:
# Split Data

from sklearn.model_selection import train_test_split

X = df_sample["Features"]
y = df_sample["Role"]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X Train Shape {x_train.shape}")
print(f"X Test Shape {x_test.shape}")

X Train Shape (40000,)
X Test Shape (10000,)


In [10]:
# Apply Tfidf

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf.fit(x_train)

x_train_tf = tfidf.transform(x_train)
x_test_tf = tfidf.transform(x_test)

In [11]:
# Train Model

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200)
rf.fit(x_train_tf, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
# Evaluate Model

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(x_test_tf)
acc = accuracy_score(y_test, y_pred)
classification_re = classification_report(y_test, y_pred)

print(f"Model Accuracy is: {acc}")
print(f"Classification Report:\n{classification_re}")

Model Accuracy is: 1.0
Classification Report:
                                precision    recall  f1-score   support

             Account Executive       1.00      1.00      1.00       164
    Administrative Coordinator       1.00      1.00      1.00       108
             Automation Tester       1.00      1.00      1.00       128
             Backend Developer       1.00      1.00      1.00       197
          Benefits Coordinator       1.00      1.00      1.00       122
 Business Intelligence Analyst       1.00      1.00      1.00       142
   Client Relationship Manager       1.00      1.00      1.00       108
               Content Creator       1.00      1.00      1.00       142
            Content Strategist       1.00      1.00      1.00       106
      Customer Success Manager       1.00      1.00      1.00       217
   Customer Support Specialist       1.00      1.00      1.00       129
                  Data Analyst       1.00      1.00      1.00       188
         Data Ent

In [13]:
# Save Model

import pickle

pickle.dump(tfidf, open("../saved_models/tfidf_recommendation.pkl", "wb"))
pickle.dump(rf, open("../saved_models/rf_recommendation.pkl", "wb"))